In [1]:
%%capture
!pip install --upgrade transformers bitsandbytes accelerate peft datasets wandb contractions spacy tqdm

In [2]:
import torch
import wandb
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
import numpy as np
import re
from sklearn.metrics import precision_score, f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from contractions import fix
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map 

# Login to Weights & Biases (W&B) - Required in Kaggle
wandb.login(key="REDACTED")  # Replace with your actual API key
wandb.init(project="Spring2025")


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dhruvjoshi892000 (dhruvjoshi892000-pace-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
# Ensure GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training")

Using cuda for training


In [4]:
import spacy

# Load spaCy model with GPU support
# spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()
    
    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available
    
    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)
    
    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/kaggle/input/15_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
# df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']


# Load tokenizer
model_name = "bert-base-uncased"  # Modify as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.10/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


spaCy is using GPU: True


Preprocessing texts: 100%|██████████| 8417/8417 [35:18<00:00,  3.97it/s]  


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/7575 [00:00<?, ? examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

In [5]:
# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=15).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/kaggle/working/logs",
    label_names=["labels"],
    logging_steps=10,
    num_train_epochs=15,  # More epochs for better legal text learning
    per_device_train_batch_size=4,  # Lower batch size for stability
    per_device_eval_batch_size=4,
    learning_rate=2e-5,  # Reduce LR for better fine-tuning
    warmup_ratio=0.1,  # Stabilize early training
    weight_decay=0.01,
    metric_for_best_model="balanced_accuracy",
    load_best_model_at_end=True,
    report_to=["wandb"],  # Enable W&B logging
    fp16=torch.cuda.is_available(),  # Use FP16 if GPU is available
)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [6]:
import random
# 🔧 Set Random Seed for Reproducibility
random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [7]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1
1,1.465100,1.163425,0.657957,0.340153,0.599951,0.657957,0.610012
2,0.748000,0.879641,0.743468,0.522862,0.722309,0.743468,0.728355
3,0.589700,0.827433,0.761283,0.593578,0.768356,0.761283,0.759707
4,0.346100,0.885582,0.766033,0.610108,0.760858,0.766033,0.752878
5,0.521200,1.087843,0.750594,0.615798,0.743490,0.750594,0.745470
6,0.771000,1.601444,0.777910,0.665706,0.780232,0.777910,0.774832
7,0.408500,2.553890,0.782660,0.617857,0.779440,0.782660,0.771613
8,0.743800,3.533306,0.763658,0.665558,0.768888,0.763658,0.762301
9,0.417000,3.790222,0.751781,0.634378,0.771123,0.751781,0.754261
10,0.442900,3.633030,0.776722,0.667569,0.777159,0.776722,0.773345


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

{'eval_loss': 3.5560011863708496, 'eval_accuracy': 0.7850356294536817, 'eval_balanced_accuracy': 0.6694846094616066, 'eval_precision': 0.7797421893493212, 'eval_recall': 0.7850356294536817, 'eval_f1': 0.7800708639800302, 'eval_runtime': 17.3888, 'eval_samples_per_second': 48.422, 'eval_steps_per_second': 6.096, 'epoch': 15.0}


In [8]:
# 279 labels

In [9]:
import spacy

# Load spaCy model with GPU support
# spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()
    
    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available
    
    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)
    
    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/kaggle/input/279_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
# df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']


# Load tokenizer
model_name = "bert-base-uncased"  # Modify as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.10/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/usr/local/lib/python3.10/dist-packages/spacy/util.py:1740: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


spaCy is using GPU: True


Preprocessing texts: 100%|██████████| 8417/8417 [39:24<00:00,  3.56it/s]  


Map:   0%|          | 0/7575 [00:00<?, ? examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

In [10]:
# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=279).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/kaggle/working/logs",
    label_names=["labels"],
    logging_steps=10,
    num_train_epochs=15,  # More epochs for better legal text learning
    per_device_train_batch_size=4,  # Lower batch size for stability
    per_device_eval_batch_size=4,
    learning_rate=2e-5,  # Reduce LR for better fine-tuning
    warmup_ratio=0.1,  # Stabilize early training
    weight_decay=0.01,
    metric_for_best_model="balanced_accuracy",
    load_best_model_at_end=True,
    report_to=["wandb"],  # Enable W&B logging
    fp16=torch.cuda.is_available(),  # Use FP16 if GPU is available
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [11]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1
1,4.797500,4.833024,0.081948,0.018626,0.052499,0.081948,0.036669
2,3.142300,3.562536,0.347981,0.141707,0.235783,0.347981,0.245445
3,2.633600,2.919964,0.446556,0.213403,0.302545,0.446556,0.341271
4,2.382600,2.624679,0.480998,0.279370,0.354902,0.480998,0.386042
5,2.380600,2.461520,0.511876,0.316047,0.402544,0.511876,0.430138
6,1.735100,2.296994,0.539192,0.350055,0.440302,0.539192,0.467594
7,1.099700,2.251248,0.568884,0.377913,0.489163,0.568884,0.506105
8,1.199300,2.264201,0.565321,0.382973,0.502118,0.565321,0.508079
9,1.113200,2.257810,0.564133,0.397326,0.504887,0.564133,0.513168
10,0.832700,2.237902,0.562945,0.390919,0.510232,0.562945,0.512094


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2184: UserWarning: y_pred

{'eval_loss': 2.2177858352661133, 'eval_accuracy': 0.5819477434679335, 'eval_balanced_accuracy': 0.427156311689389, 'eval_precision': 0.5428557880248666, 'eval_recall': 0.5819477434679335, 'eval_f1': 0.541140719674231, 'eval_runtime': 17.1478, 'eval_samples_per_second': 49.103, 'eval_steps_per_second': 6.182, 'epoch': 15.0}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2184: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
